## Data Cleaning for Ag slab w/ [C, H, O] adsorbates
##### DropBox or Box --> filtered folder names

In [1]:
import os
import time
from dotenv import load_dotenv

load_dotenv(override=True)

True

### Get files from DropBox

In [ ]:
import dropbox
from dropbox.files import SharedLink

DB_TOKEN = os.getenv("DROPBOX_TOKEN")
DB_APP_KEY = os.getenv("APP_KEY")
DB_APP_SECRET = os.getenv("APP_SECRET")
DB_REFRESH_TOKEN = os.getenv("REFRESH_TOKEN")

One-time DropBox refresh token authentication (don't need to rerun this)

In [ ]:
from dropbox.oauth import DropboxOAuth2FlowNoRedirect

auth_flow = DropboxOAuth2FlowNoRedirect(
    consumer_key=DB_APP_KEY,
    consumer_secret=DB_APP_SECRET,
    token_access_type="offline",
)

print(auth_flow.start())

https://www.dropbox.com/oauth2/authorize?response_type=code&client_id=jwuqsuv0crfxtab&token_access_type=offline


In [9]:
code = "" 

oauth_result = auth_flow.finish(code)
print(oauth_result.refresh_token)

aliMnF1simkAAAAAAAAAASGil5pP5_yYUBsywliZYcR9iwC6t7aXl9z5s_cg48Om


In [ ]:
# access dropbox API

dbx = dropbox.Dropbox(
    oauth2_refresh_token=DB_REFRESH_TOKEN,
    app_key=DB_APP_KEY,
    app_secret=DB_APP_SECRET,
)

ROOT_PATH = SharedLink(url="https://www.dropbox.com/scl/fo/qpg1zuo3g7vb3il1wmqy3/AA4wERzz28lJhYTCAcSEvqk?dl=0")

In [ ]:
# fetch list of all subfolders in 1st level of DropBox folder

results = []
def list_folder():
    res = dbx.files_list_folder(path="", shared_link=ROOT_PATH)
    
    while True:
        for entry in res.entries:
            results.append({
                "name": entry.name,
                "type": entry.__class__.__name__
            })
        if not res.has_more:
            break
        res = dbx.files_list_folder_continue(res.cursor)

list_folder()
print(f"Total items: {len(results)}")

# write txt file of folder names
with open("cache/dropbox_foldernames.txt", "w", encoding="utf-8") as f:
    for entry in results:
        if entry["type"] == "FolderMetadata":
            f.write(entry["name"] + "\n")

print("Saved to cache/dropbox_foldernames.txt")

### Get files from Box

In [4]:
from box_sdk_gen import BoxClient, BoxDeveloperTokenAuth
load_dotenv(override=True)

BOX_DEV_TOKEN = os.getenv("DEVELOPER_TOKEN")
BOX_FOLDER_IDS = ["376580648972", "376581493469"]
# EO_Project_reactivity, EO_Project_reactivity_part2

auth = BoxDeveloperTokenAuth(BOX_DEV_TOKEN)
client = BoxClient(auth)

In [5]:
# check number of first-level subfolders

for folder_id in BOX_FOLDER_IDS:
    info = client.folders.get_folder_by_id(folder_id, fields=["item_collection", "name"])
    print(f"folder id: {folder_id}, info.name, folder count: {info.item_collection.total_count}")

folder id: 376580648972, info.name, folder count: 295
folder id: 376581493469, info.name, folder count: 76


In [7]:
# get all folder names and write list to cache
allnames = set()

for folder_id in BOX_FOLDER_IDS:
    offset = 0
    limit = 1000

    while True:
        items = client.folders.get_folder_items(
            folder_id,
            limit=limit,
            offset=offset,
        )

        for item in items.entries:
            if item.type == "folder":
                allnames.add(item.name)

        offset += len(items.entries)
        if offset >= items.total_count:
            break

with open("cache/EOreactivity_foldernames.txt", "w", encoding="utf-8") as f:
    for name in sorted(allnames):
        f.write(name + "\n")

print(f"Saved {len(allnames)} folder names")

Saved 369 folder names


## Filter through folder names

In [8]:
# read in raw folder names list from cache (change file as needed)
#     from DropBox: dropbox_foldernames.txt
with open("cache/EOreactivity_foldernames.txt", "r", encoding="utf-8") as f:
    foldernames = [line.strip() for line in f]

# filter out obviously bad data via substring matching
exclude = {
    "wrong",
    "toomanykpoints",
    "oldbad",
    "exploded",
        }

foldernames = [n for n in foldernames if not any(term in n for term in exclude)]
print(f"Total items: {len(foldernames)}")

Total items: 369


In [9]:
# filter out elements and calculation types we don't want
exclude = {
    "Al", "Au", "Co", "Cr", "Cu", "Fe", "Ga", "In", "Ir", "Mg", "Mn", "Mo", "Pd", "Pt", "Re", "Rh", "Ru", "Sc", "Sn", "Ti", "V", "W", "Zn"
    "minhop", "CL", "CL_JS", "Bader", "Bare", "bare", "Bulk", "Gas", "gas", "TS",
}

folders = []

# include names w/ "Ni0" substring (means zero Ni)
for name in foldernames:
    if "Ni" in name and "Ni0" not in name:
        continue
    if any(term in name for term in exclude):
        continue
    folders.append(name)

foldernames = folders

print(f"Total items: {len(foldernames)}")

Total items: 231


### for DropBox

In [9]:
# check if OUTCAR exists and if so, get elements from POSCAR

elements_map = {}
irregular_poscars = {}
no_outcar = []
failed_folders = {}

for folder in foldernames:
    folder_path = f"/{folder}"
    try:
        res = dbx.files_list_folder(path=folder_path, shared_link=ROOT_PATH)
        names = {e.name for e in res.entries}
    except dropbox.exceptions.ApiError as e:
        failed_folders[folder] = str(e)
        continue

    if "OUTCAR" not in names:
        no_outcar.append(folder)
        continue

    try:
        _, dres = dbx.sharing_get_shared_link_file(url=ROOT_PATH.url, path=f"{folder_path}/POSCAR")
        lines = dres.content.decode("utf-8", errors="ignore").splitlines()
        tokens = lines[5].split()
        if all(t.isalpha() for t in tokens):
            elements_map[folder] = set(tokens)
        else:
            irregular_poscars[folder] = lines[:5]
    except dropbox.exceptions.ApiError as e:
        failed_folders[folder] = str(e)

    time.sleep(0.2)

### for Box

In [ ]:
# get all folder names+ids via Box API and store in set

subfolder_map = {}  # name -> folder_id
for folder_id in BOX_FOLDER_IDS:
    offset = 0
    limit = 1000
    while True:
        items = client.folders.get_folder_items(folder_id, limit=limit, offset=offset)
        for item in items.entries:
            if item.type == "folder":
                subfolder_map[item.name] = item.id
        offset += len(items.entries)
        if offset >= items.total_count:
            break

In [ ]:
from box_sdk_gen import BoxAPIError

elements_map = {}
irregular_poscars = {}
no_outcar = []
failed_folders = {}

for folder, folder_id in subfolder_map.items():
    try:
        items = client.folders.get_folder_items(folder_id, limit=1000)
        names = {e.name: e.id for e in items.entries}
    except BoxAPIError as e:
        failed_folders[folder] = str(e)
        continue

    if "OUTCAR" not in names:
        no_outcar.append(folder)
        continue

    try:
        file_stream = client.downloads.download_file(names["POSCAR"])
        content = file_stream.read()
        lines = content.decode("utf-8", errors="ignore").splitlines()
        tokens = lines[5].split()
        if all(t.isalpha() for t in tokens):
            elements_map[folder] = set(tokens)
        else:
            irregular_poscars[folder] = lines[:5]
    except BoxAPIError as e:
        failed_folders[folder] = str(e)
    time.sleep(0.2)

In [ ]:
print(f"""
    elements_map: {len(elements_map)} items
    irregular_poscars: {len(irregular_poscars)} items
    no_outcar": {len(no_outcar)} items
    failed_folders: {len(failed_folders)} items
    """)

In [ ]:
# inspect irregular POSCARs

for folder, lines in irregular_poscars.items():
    print(folder)
    print("\n".join(lines))
    print("-"*10)

Saved to cache/cleaned_foldernames.txt


In [17]:
allowed = {"Ag", "C", "H", "O"}
clean_map = {k: v for k, v in elements_map.items() if v.issubset(allowed)}

print(f"Total items: {len(clean_map)}")

Total items: f163, Missing INCARs: 0


In [18]:
# last round of misc. filtering from manual inspection

exclude = {"TS", "spin"}
clean_map = {
    k: v for k, v in clean_map.items()
    if v.issubset(allowed) and not any(sub in k for sub in exclude)
}

# delete folder if folder_good exists
suffixed = {k[:-5] for k in clean_map if k.endswith("_good")}
clean_map = {k: v for k, v in clean_map.items() if k not in suffixed}

print(f"Total items: {len(clean_map)}")

Total items: 143


In [19]:
# write txt file of cleaned folder names
with open("cache/cleaned_foldernames.txt", "w", encoding="utf-8") as f:
    for entry in clean_map:
        f.write(entry + "\n")

print("Saved to cache/cleaned_foldernames.txt")

Saved to cache/selected_foldernames.txt


# get INCARs for clean folders

incar_map = {}
missing_incars = []

for folder in clean_map:
    try:
        _, res = dbx.sharing_get_shared_link_file(url=ROOT_PATH.url, path=f"/{folder}/INCAR")
        incar_map[folder] = Incar.from_str(res.content.decode("utf-8", errors="ignore"))
    except dropbox.exceptions.ApiError:
        missing_incars.append(folder)
    time.sleep(0.2)

print(f"Total items: f{len(incar_map)}, Missing INCARs: {len(missing_incars)}")

In [2]:
# filter out DFT calculations other than relaxations

filtered_incar_map = {
    k: v for k, v in incar_map.items()
    if v.get("IBRION") in (1, 2, 3) and v.get("ISPIN", 1) == 1 and v.get("NSW", 0) != 0
}
print(f"Total items: {len(filtered_incar_map)}")

NameError: name 'exclude' is not defined

# write txt file of cleaned and selected folder names (relaxations)
with open("cache/selected_foldernames.txt", "w", encoding="utf-8") as f:
    for entry in filtered_incar_map:
        f.write(entry + "\n")

print("Saved to cache/selected_foldernames.txt")

In [1]:
continue data preparation in `ag-data-selection.ipynb`

NameError: name 'foldernames' is not defined

# if needed for another round of filtering: read in raw folder names list from cache

with open("cache/selected_foldernames.txt", "r", encoding="utf-8") as f:
    foldernames = [line.strip() for line in f]

foldernames = [v for v in foldernames if not any(sub in v for sub in exclude)]

print(f"Total items: {len(foldernames)}")

In [ ]:
Check edge cases (runs that made it into 'selected' but not 'screened')

In [ ]:
with open("cache/screened_foldernames.txt", "r", encoding="utf-8") as f:
    screenednames =  [line.strip() for line in f]

edgecases = [v for v in foldernames if v not in screenednames]

print(f"Total items: {len(edgecases)}")